In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from zipfile import ZipFile

In [ ]:
zip_path = "/content/drive/MyDrive/seg_test.zip"

extract_path = "/content/"

with ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction Completed!")

In [ ]:
zip_path = "/content/drive/MyDrive/seg_train.zip"

extract_path = "/content/"

with ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction Completed!")

In [ ]:
import os

os.listdir("/content/")

In [ ]:
import tensorflow as tf

test_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/seg_test",
    image_size=(150,150),
    batch_size=32,
    shuffle=False
)

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/seg_train",
    image_size=(150,150),
    batch_size=32,
    shuffle=True,
    label_mode="int",
    seed=42
)

In [ ]:
class_names = train_ds.class_names

print(class_names)

In [ ]:
for images, labels in train_ds.take(1):

    print("Images Shape :", images.shape)

    print("Labels Shape :", labels.shape)

In [ ]:
plt.figure(figsize=(12,12))

for images, labels in train_ds.take(1):

    for i in range(9):

        plt.subplot(3,3,i+1)

        plt.imshow(images[i].numpy().astype("uint8"))

        plt.title(class_names[labels[i]])

        plt.axis("off")

plt.show()

In [ ]:
plt.figure(figsize=(12,12))

for images, labels in test_ds.take(1):

    for i in range(9):

        plt.subplot(3,3,i+1)

        plt.imshow(images[i].numpy().astype("uint8"))

        plt.title(class_names[labels[i]])

        plt.axis("off")

plt.show()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Rescaling,
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)


In [ ]:
model = Sequential()
model.add(Input(shape=(150, 150, 3)))
model.add(Rescaling(1./255))

In [ ]:
model.add(
    Conv2D(
        filters=32,
        kernel_size=(3,3),
        activation="relu"
    )
)
model.add(
    MaxPooling2D(
        pool_size=(2,2)
    )
)

In [ ]:
model.add(
    Conv2D(
        filters=64,
        kernel_size=(3,3),
        activation="relu"
    )
)

model.add(
    MaxPooling2D(
        pool_size=(2,2)
    )
)

In [ ]:
model.add(
    Conv2D(
        filters=128,
        kernel_size=(3,3),
        activation="relu"
    )
)

model.add(
    MaxPooling2D(
        pool_size=(2,2)
    )
)

In [ ]:
model.add(Flatten())
model.add(
    Dense(
        units=128,
        activation="relu"
    )
)
model.add(
    Dropout(0.5)
)
model.add(
    Dense(
        units=6,
        activation="softmax"
    )
)

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=15
)

In [ ]:
loss, accuracy = model.evaluate(test_ds)

print("Loss :", loss)
print("Accuracy :", accuracy)

In [ ]:
y_pred_prob = model.predict(test_ds)

In [ ]:
y_pred = np.argmax(y_pred_prob, axis=1)

In [ ]:
y_true = np.concatenate(
    [labels.numpy() for images, labels in test_ds]
)

In [ ]:
print(y_true.shape)
print(y_pred.shape)

In [ ]:

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
accuracy = accuracy_score(y_true, y_pred)

print(f"Accuracy : {accuracy:.4f}")

In [ ]:
print(classification_report(
    y_true,
    y_pred,
    target_names=train_ds.class_names
))


In [ ]:
cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=train_ds.class_names)
disp.plot(cmap="Blues")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

for images, labels in test_ds.take(1):

    predictions = model.predict(images)

    predicted_classes = np.argmax(predictions, axis=1)

    plt.figure(figsize=(15,10))

    for i in range(9):

        plt.subplot(3,3,i+1)

        plt.imshow(images[i].numpy().astype("uint8"))

        plt.title(
            f"Pred: {train_ds.class_names[predicted_classes[i]]}\n"
            f"True: {train_ds.class_names[labels[i]]}"
        )

        plt.axis("off")

    plt.tight_layout()
    plt.show()

    break

# **Data Augmentation**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Rescaling,
    RandomFlip,
    RandomRotation,
    RandomZoom,
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)

In [ ]:
data_augmentation = Sequential()

data_augmentation.add(RandomFlip("horizontal"))

data_augmentation.add(RandomRotation(0.1))

data_augmentation.add(RandomZoom(0.2))

In [ ]:
model1 = Sequential()

model1.add(Input(shape=(150,150,3)))

model1.add(data_augmentation)

model1.add(Rescaling(1./255))

model1.add(Conv2D(32,(3,3),activation="relu"))
model1.add(MaxPooling2D((2,2)))

model1.add(Conv2D(64,(3,3),activation="relu"))
model1.add(MaxPooling2D((2,2)))

model1.add(Conv2D(128,(3,3),activation="relu"))
model1.add(MaxPooling2D((2,2)))

model1.add(Flatten())

model1.add(Dense(128,activation="relu"))

model1.add(Dropout(0.5))

model1.add(Dense(6,activation="softmax"))

In [ ]:
model1.summary()

In [ ]:
model1.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history1 = model1.fit(
    train_ds,
    validation_data=test_ds,
    epochs=15
)

In [ ]:
loss, accuracy = model1.evaluate(test_ds)

print(f"Loss     : {loss:.4f}")
print(f"Accuracy : {accuracy:.4f}")

In [ ]:
y_pred_prob = model1.predict(test_ds)

y_pred1 = np.argmax(y_pred_prob, axis=1)

In [ ]:
y_true = np.concatenate(
    [labels.numpy() for _, labels in test_ds]
)

In [ ]:
accuracy = accuracy_score(y_true, y_pred1)

print(f"Accuracy : {accuracy*100:.2f}%")

In [ ]:
print("\nClassification Report\n")

print(
    classification_report(
        y_true,
        y_pred1,
        target_names=train_ds.class_names
    )
)

# **With Batch Normalization**

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Rescaling,
    RandomFlip,
    RandomRotation,
    RandomZoom,
    Conv2D,
    BatchNormalization,
    Activation,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)

In [ ]:
data_augmentation = Sequential()

data_augmentation.add(RandomFlip("horizontal"))
data_augmentation.add(RandomRotation(0.1))
data_augmentation.add(RandomZoom(0.2))

In [ ]:
model2 = Sequential()

model2.add(Input(shape=(150,150,3)))

model2.add(data_augmentation)

model2.add(Rescaling(1./255))

In [ ]:
model2.add(Conv2D(32,(3,3),padding="same"))

model2.add(BatchNormalization())

model2.add(Activation("relu"))

model2.add(MaxPooling2D((2,2)))

In [ ]:
model2.add(Conv2D(64,(3,3),padding="same"))

model2.add(BatchNormalization())

model2.add(Activation("relu"))

model2.add(MaxPooling2D((2,2)))


In [ ]:
model2.add(Conv2D(128,(3,3),padding="same"))

model2.add(BatchNormalization())

model2.add(Activation("relu"))

model2.add(MaxPooling2D((2,2)))

In [ ]:
model2.add(Flatten())

model2.add(Dense(128))

model2.add(Activation("relu"))

model2.add(Dropout(0.5))

model2.add(Dense(6, activation="softmax"))

In [ ]:
model2.summary()

In [ ]:
model2.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history2 = model2.fit(
    train_ds,
    validation_data=test_ds,
    epochs=15
)

In [ ]:
loss, accuracy = model2.evaluate(test_ds)

print(f"\nTest Loss     : {loss:.4f}")
print(f"Test Accuracy : {accuracy:.4f}")

In [ ]:
y_pred_prob = model2.predict(test_ds)

y_pred = np.argmax(y_pred_prob, axis=1)


In [ ]:
print(f"{accuracy_score(y_true,y_pred):.4f}")


In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=train_ds.class_names
    )
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_accuracy",
    mode="max",
    patience=5,
    restore_best_weights=True,
    verbose=1
)